In [1]:
!pip install langchain langchain-community langchain-core langchain-google-genai langchain-text-splitters faiss-cpu pypdf tiktoken

In [2]:
import os, re, time, getpass
os.environ["GEMINI_API_KEY"] = getpass.getpass("Enter your Gemini API key: ")

Enter your Gemini API key: ··········


In [3]:
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings

llm = ChatGoogleGenerativeAI(model="gemini-3.8-flash", temperature=0)
embedding = GoogleGenerativeAIEmbeddings(model="gemini-embedding-001")

In [4]:
from langchain_community.document_loaders import PyPDFLoader

pages = PyPDFLoader("/content/intro-to-ml-book.pdf").load()
print("Pages loaded:", len(pages))

/tmp/ipykernel_9798/3039986001.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


Pages loaded: 392


In [5]:
all_text = "\n".join(p.page_content for p in pages)

print("Number of pages:", len(pages))
print("Empty pages (under 20 characters):", sum(1 for p in pages if len(p.page_content.strip()) < 20))
print("Words broken across lines by a hyphen:", len(re.findall(r"\w-\n\w", all_text)))
print("Page footers such as 'Title | 37':", len(re.findall(r"\|\s*\d+\s*$", all_text, flags=re.M)))
print("Book title repeated on pages:", all_text.count("Introduction to Machine Learning with Python"))

Number of pages: 392
Empty pages (under 20 characters): 5
Words broken across lines by a hyphen: 56
Page footers such as 'Title | 37': 179
Book title repeated on pages: 2


In [6]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    encoding_name="cl100k_base",
    chunk_size=700,
    chunk_overlap=100,
)
chunks = splitter.split_documents(pages)
print("Total chunks:", len(chunks))

Total chunks: 410


In [7]:
from langchain_community.vectorstores import FAISS

vector_store = None
BATCH_SIZE = 40

for start in range(0, len(chunks), BATCH_SIZE):
    batch = chunks[start:start + BATCH_SIZE]
    if vector_store is None:
        vector_store = FAISS.from_documents(batch, embedding)
    else:
        vector_store.add_documents(batch)
    print("Embedded", min(start + BATCH_SIZE, len(chunks)), "of", len(chunks))
    time.sleep(40)

print("Vectors stored in FAISS:", vector_store.index.ntotal)

Embedded 40 of 410
Embedded 80 of 410
Embedded 120 of 410
Embedded 160 of 410
Embedded 200 of 410
Embedded 240 of 410
Embedded 280 of 410


GoogleGenerativeAIError: Error embedding content (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://aistudio.google.com/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/embed_content_free_tier_tokens, limit: 30000, model: gemini-embedding-1.0, description: Embed content tokens per minute limit exceeded on Free Tier. See https://aistudio.google.com/docs/rate-limits.\nPlease retry in 29.005349352s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://aistudio.google.com/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'description': 'Embed content tokens per minute 

In [8]:
sample_query = "What is overfitting?"

query_vector = embedding.embed_query(sample_query)
results = vector_store.similarity_search_by_vector(query_vector, k=3)

for doc in results:
    print("Page", doc.metadata["page"] + 1)
    print(doc.page_content[:300], "...")
    print()

Page 42
3 And also provably, with the right math.
58 years old want to buy a boat, while all others don’t. While we can make up many
rules that work well on this data, remember that we are not interested in making pre‐
dictions for this dataset; we already know the answers for these customers. We want
to kn ...

Page 41
2 In the real world, this is actually a tricky problem. While we know that the other customers haven’t bought a
boat from us yet, they might have bought one from someone else, or they may still be saving and plan to buy
one in the future.
Usually we build a model in such a way that it can make accur ...

Page 62
You might notice the strange-looking trailing underscore at the end
of coef_ and intercept_. scikit-learn always stores anything
that is derived from the training data in attributes that end with a
trailing underscore. That is to separate them from parameters that
are set by the user.
The intercept_ ...



In [9]:
def retrieve(question, k=5):
    query_vector = embedding.embed_query(question)            # query -> embedding
    return vector_store.similarity_search_by_vector(query_vector, k=k)   # top-k chunks

In [10]:
experiment_query = "What is overfitting?"

for k in [3, 5, 7]:
    docs = retrieve(experiment_query, k=k)
    pages_found = sorted({d.metadata["page"] + 1 for d in docs})
    print(f"k={k}: {len(docs)} chunks, from pages {pages_found}")

k=3: 3 chunks, from pages [41, 42, 62]
k=5: 5 chunks, from pages [41, 42, 43, 53, 62]
k=7: 7 chunks, from pages [41, 42, 43, 53, 62, 63, 73]


In [11]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

prompt = ChatPromptTemplate.from_template("""
You are a machine learning tutor.
Answer the question using ONLY the context below. Do not use any other knowledge.
If the answer is not in the context, reply exactly: "I couldn't find this in the book."
After each point, cite the page number, for example (p. 42).

Context:
{context}

Question: {question}
""")

chain = prompt | llm | StrOutputParser()

In [12]:
def answer(question, k=5):
    docs = retrieve(question, k=k)
    context = "\n\n".join(f"[p. {d.metadata['page'] + 1}] {d.page_content}" for d in docs)
    response = chain.invoke({"context": context, "question": question})

    print("Question:", question)
    print("Answer:", response)
    print("Pages used:", sorted({d.metadata["page"] + 1 for d in docs}))
    print("\nRetrieved context:")
    for d in docs:
        print(f"  [p. {d.metadata['page'] + 1}] {d.page_content[:150]}...")
    print("-" * 80)

In [13]:
demo_questions = [
    "What is overfitting and how can we detect it?",
    "How does the k-nearest neighbors algorithm make a prediction?",
    "What does one-hot encoding do?",
    "What is cross-validation?",
    "What is the capital of France?",
]

for q in demo_questions:
    answer(q, k=5)

Question: What is overfitting and how can we detect it?
Answer: Based on the provided text, here is what overfitting is and how it can be detected:

**What is overfitting?**
* Overfitting occurs when a model is built to be too complex for the amount of information available (p. 42).
* It happens when you fit a model too closely to the particularities of the training set, producing a model that performs well on the training data but fails to generalize to new data (p. 42). 

**How can we detect it?**
* The only measure of whether an algorithm will perform well on new data is evaluation on the test set (p. 42).
* You can detect it by comparing performance on the training set and the test set; overfitting is indicated when the model predicts very accurately on the training set, but performs significantly worse on the test set (p. 62).
Pages used: [41, 42, 53, 62, 73]

Retrieved context:
  [p. 42] 3 And also provably, with the right math.
58 years old want to buy a boat, while all others d

In [14]:
while True:
    question = input("Your question: ")
    if question.strip().lower() == "exit":
        print("Goodbye!")
        break
    answer(question, k=5)

Your question: What is machine learning?
Question: What is machine learning?
Answer: Based on the context, here is what machine learning is:

* Machine learning is about extracting knowledge from data (p. 15).
* It is a research field at the intersection of statistics, artificial intelligence, and computer science, and is also known as predictive analytics or statistical learning (p. 15).
* The most successful kinds of machine learning algorithms automate decision-making processes by generalizing from known examples (p. 16).
Pages used: [9, 15, 16, 17, 18]

Retrieved context:
  [p. 16] spam. This would be an example of using an expert-designed rule system to design an
“intelligent” application. Manually crafting decision rules is fea...
  [p. 15] CHAPTER 1
Introduction
Machine learning is about extracting knowledge from data. It is a research field at the
intersection of statistics, artificial ...
  [p. 18] Segmenting customers into groups with similar preferences
Given a set of custom